In [100]:
import contextlib
import importlib.util
import os
import sys
from itertools import cycle
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from orbit_wars_engine import OrbitWarsEngine
from orbit_wars_model import encode_obs

In [65]:
NUM_FRAMES = 4
ACTIONS_DIM = 6
SEND_FRACTIONS = (0.25, 0.50, 0.75, 1.00)
CONST_SEND = 42
TOKEN_DIM = 9
GLOBAL_DIM = 16 

In [5]:
env = OrbitWarsEngine(num_players=2)

In [101]:
seed = np.random.randint(0, 100_000_000)
obs = env.reset(seed)['observations']

In [43]:
encoded_obs = encode_obs(obs[0], 0)  # for player 0
encoded_obs.keys()

dict_keys(['planet_ids', 'num_planets', 'frame_offsets', 'frame_planets', 'tokens', 'tokens_shape', 'globals', 'globals_shape', 'presence', 'presence_shape', 'turns', 'turns_shape', 'angles', 'angles_shape', 'mask', 'mask_shape'])

In [126]:
global_features = torch.from_numpy(encoded_obs['globals'])                                    # (16,)       f32 
planet_tokens = torch.from_numpy(encoded_obs['tokens'].reshape(NUM_FRAMES, 44, TOKEN_DIM))    # (4, 44, 6)  f32
planet_presence = torch.from_numpy(encoded_obs['presence'].reshape(NUM_FRAMES, 44))           # (4, 44)     f32
distances = torch.from_numpy(encoded_obs['turns'].reshape(44, 44, ACTIONS_DIM))               # (44, 44, 6) f32
valid_actions_mask = torch.from_numpy(encoded_obs['mask'].reshape(44, 44, ACTIONS_DIM))       # (44, 44, 6) u8
launch_angles = torch.from_numpy(encoded_obs['angles'].reshape(44, 44, ACTIONS_DIM))          # (44, 44, 6) f32

In [375]:
class PlanetEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.mlp = nn.Sequential(  # (44, TOKEN_DIM) -> (44, 64)
            nn.Linear(TOKEN_DIM, 64),
            nn.SiLU(),
            nn.Linear(64, 64),
        )
        
    def forward(self, planet_tokens):
        return self.mlp(planet_tokens)
        

class GNNLayer(nn.Module):
    def __init__(self):
        super().__init__()

        # input is [travel times ~ reach bits]; encodes a message
        # (44, 44, 2 * ACTIONS_DIM) -> (44, 44, 64)
        self.phi = nn.Sequential(
            nn.Linear(2 * ACTIONS_DIM, 64),
            nn.SiLU(),
            nn.Linear(64, 64),  # keep linear as last to allow representation of negatives
        )

        # transforms summed messages into a planet update
        # (44, 64) -> (44, 64)
        self.rho = nn.Sequential(
            nn.Linear(64, 64),
            nn.SiLU(),
            nn.Linear(64, 64),
        )

    def forward(
        self,
        h,                   # (44, 64) of encoded tokens after passing through PlanetEncoder
        distances,           # (44, 44, ACTIONS_DIM)
    ) -> torch.Tensor:       # (44, 64)
        
        with torch.no_grad():
            d_ji = distances.transpose(0, 1)              # (44, 44, ACTIONS_DIM)
            reach_ji = (d_ji > 0).float()                 # (44, 44, ACTIONS_DIM)
            edge_mask = reach_ji.any(dim=-1)              # (44, 44) bitplane that is True if any action can be taken
            
        phi_in = torch.cat([d_ji, reach_ji], dim=-1)  # (44, 44, 2 * ACTIONS_DIM)
        msgs = gnn_layer.phi(phi_in)                  # (44, 44, 64)
        msgs = msgs * edge_mask.unsqueeze(-1)         # (44, 44, 64) kills non edges
        agg = msgs.sum(dim=1)                         # (44, 44) sum over senders dim=1, sum is permutation invariant
        h_out = h + gnn_layer.rho(agg)                # (44, 64) residual update
        return h_out


class FrameFusion(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = nn.Linear(NUM_FRAMES * 64, 64)  # (NUM_FRAMES * 64,) -> (64,)

    def forward(self, joined):
        x = joined.permute(1, 0, 2).reshape(44, NUM_FRAMES * 64)
        return self.proj(x)


class GlobalEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.mlp = nn.Sequential(  # (GLOBAL_DIM,) -> (64,)
            nn.Linear(GLOBAL_DIM, 64),
            nn.SiLU(),
            nn.Linear(64, 64),
        )
        
    def forward(self, global_features):
        return self.mlp(global_features)


class ValueHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.planet_proj = nn.Sequential(
            nn.Linear(64, 64),
            nn.SiLU(),
            nn.Linear(64, 64),
        )
        self.mlp = nn.Sequential(
            nn.Linear(64 + 64, 64),
            nn.SiLU(),
            nn.Linear(64, 1),
        )

        
    def forward(self, fused_planets, encoded_globals, planet_presence):
        p = planet_presence[0].unsqueeze(-1)
        planets_projection = self.planet_proj(fused_planets)
        masked_planets = p * planets_projection
        pooled = masked_planets.sum(0) / p.sum().clamp(min=1)
        with_globals = torch.cat([pooled, encoded_globals])
        out = self.mlp(with_globals)
        return out


class PolicyHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.planet_proj = nn.Sequential(
            nn.Linear(64, 64),
            nn.SiLU(),
            nn.Linear(64, 64),
        )
        self.src_proj = nn.Sequential(
            nn.Linear(64 + 64, 64),   # Claude said this is similar to attention
            nn.SiLU(),
            nn.Linear(64, 64),
        )
        self.tgt_proj = nn.Sequential(
            nn.Linear(64, 64),        # ...
            nn.SiLU(),
            nn.Linear(64, 64),
        )
        self.action_mlp = nn.Sequential(
            nn.Linear(64 + 64, 64),
            nn.SiLU(),
            nn.Linear(64, ACTIONS_DIM),
        )

        
    def forward(self, fused_planets, encoded_globals, valid_actions_mask):
        h = self.planet_proj(fused_planets)
        g = encoded_globals.expand(44, -1)
        h_src = self.src_proj(torch.cat([h, g], dim=-1))  # (44, 64)
        h_tgt = self.tgt_proj(h)                          # (44, 64)
        
        # make sure to unsqueeze to disambiguate otherwise, dims are prepended
        src = h_src.unsqueeze(1).expand(44, 44, -1)                             # (44, 44, 64)  
        tgt = h_tgt.unsqueeze(0).expand(44, 44, -1)                             # (44, 44, 64)
        logits = self.action_mlp(torch.cat([src, tgt], dim=-1))                 # (44, 44, ACTIONS_DIM)
        
        logits = logits.masked_fill(~valid_actions_mask.bool(), float('-inf'))  # (44, 44, ACTIONS_DIM)
        probs = F.softmax(logits.view(44, 44 * ACTIONS_DIM), dim=-1)            # (44, 44 * ACTIONS_DIM)
        
        # probs includes nans - remember to mask properly in loss
        return logits, probs


class GalaxyNet(nn.Module):
    def __init__(self):
        super().__init__()
        planet_encoder = PlanetEncoder()
        gnn_layer = GNNLayer()
        frame_fusion = FrameFusion()
        global_encoder = GlobalEncoder()
        value_head = ValueHead()
        policy_head = PolicyHead()

    def forward(
        self, 
        global_features,      # (16,)       f32 
        planet_tokens,        # (4, 44, 6)  f32
        planet_presence,      # (4, 44)     f32
        distances,            # (44, 44, 6) f32
        valid_actions_mask,   # (44, 44, 6) u8
        *,
        do_value=True,
        do_policy=True,
    ):
        encoded_planets = planet_encoder(planet_tokens)
        gnn_out = gnn_layer(encoded_planets[0], distances)
        joined = torch.cat([gnn_out.unsqueeze(0), encoded_planets[1:]], dim=0)
        fused = frame_fusion(joined)
        encoded_globals = global_encoder(global_features)

        ret = []
        
        if do_value:
            value = value_head(fused, encoded_globals, planet_presence)
            ret.append(value)

        if do_policy:
            logits, probs = policy_head(fused, encoded_globals, valid_actions_mask)
            ret.extend([logits, probs])

        return ret

In [376]:
galaxy_net = GalaxyNet()
value, logits, probs = galaxy_net(global_features, planet_tokens, planet_presence, distances, valid_actions_mask)
galaxy_net(global_features, planet_tokens, planet_presence, distances, valid_actions_mask, do_value=False)

[tensor([[[-inf, -inf, -inf, -inf, -inf, -inf],
          [-inf, -inf, -inf, -inf, -inf, -inf],
          [-inf, -inf, -inf, -inf, -inf, -inf],
          ...,
          [-inf, -inf, -inf, -inf, -inf, -inf],
          [-inf, -inf, -inf, -inf, -inf, -inf],
          [-inf, -inf, -inf, -inf, -inf, -inf]],
 
         [[-inf, -inf, -inf, -inf, -inf, -inf],
          [-inf, -inf, -inf, -inf, -inf, -inf],
          [-inf, -inf, -inf, -inf, -inf, -inf],
          ...,
          [-inf, -inf, -inf, -inf, -inf, -inf],
          [-inf, -inf, -inf, -inf, -inf, -inf],
          [-inf, -inf, -inf, -inf, -inf, -inf]],
 
         [[-inf, -inf, -inf, -inf, -inf, -inf],
          [-inf, -inf, -inf, -inf, -inf, -inf],
          [-inf, -inf, -inf, -inf, -inf, -inf],
          ...,
          [-inf, -inf, -inf, -inf, -inf, -inf],
          [-inf, -inf, -inf, -inf, -inf, -inf],
          [-inf, -inf, -inf, -inf, -inf, -inf]],
 
         ...,
 
         [[-inf, -inf, -inf, -inf, -inf, -inf],
          [-inf, -

In [367]:
planet_encoder = PlanetEncoder()
gnn_layer = GNNLayer()
frame_fusion = FrameFusion()
global_encoder = GlobalEncoder()
value_head = ValueHead()
policy_head = PolicyHead()

encoded_planets = planet_encoder(planet_tokens)
gnn_out = gnn_layer(encoded_planets[0], distances)
joined = torch.cat([gnn_out.unsqueeze(0), encoded_planets[1:]], dim=0)
fused = frame_fusion(joined)
encoded_globals = global_encoder(global_features)

value = value_head(fused, encoded_globals, planet_presence)
logits, probs = policy_head(fused, encoded_globals, valid_actions_mask)

In [357]:
fused_planets = fused
h = self.planet_proj(fused_planets)
g = encoded_globals.expand(44, -1)
h_src = self.src_proj(torch.cat([h, g], dim=-1))  # (44, 64)
h_tgt = self.tgt_proj(h)                          # (44, 64)

# make sure to unsqueeze to disambiguate otherwise, dims are prepended
src = h_src.unsqueeze(1).expand(44, 44, -1)                             # (44, 44, 64)  
tgt = h_tgt.unsqueeze(0).expand(44, 44, -1)                             # (44, 44, 64)
logits = self.action_mlp(torch.cat([src, tgt], dim=-1))                 # (44, 44, ACTIONS_DIM)

logits = logits.masked_fill(~valid_actions_mask.bool(), float('-inf'))  # (44, 44, ACTIONS_DIM)
probs = F.softmax(logits.view(44, 44 * ACTIONS_DIM), dim=-1)

probs  # includes nans - remember to mask properly in loss

tensor([[nan, nan, nan,  ..., nan, nan, nan],
        [nan, nan, nan,  ..., nan, nan, nan],
        [nan, nan, nan,  ..., nan, nan, nan],
        ...,
        [nan, nan, nan,  ..., nan, nan, nan],
        [nan, nan, nan,  ..., nan, nan, nan],
        [nan, nan, nan,  ..., nan, nan, nan]], grad_fn=<SoftmaxBackward0>)

In [338]:
h = torch.randn(44, 64)

h.expand(44, 44, -1)          # works for me!

tensor([[[ 1.0265, -0.5188, -0.1022,  ...,  0.4857, -0.2457, -0.3435],
         [-0.0531,  1.0216, -0.2858,  ..., -0.1946,  0.3140,  0.2594],
         [ 0.4319, -0.5107,  0.7150,  ..., -0.3964,  0.4322, -0.0792],
         ...,
         [ 1.7345,  0.8501, -0.4626,  ..., -0.2404,  0.8107,  1.1856],
         [-1.7627, -1.0018,  1.2368,  ...,  0.7324, -1.3661,  0.5996],
         [-0.9521, -0.4527, -0.6780,  ..., -1.6143,  0.4245, -1.4192]],

        [[ 1.0265, -0.5188, -0.1022,  ...,  0.4857, -0.2457, -0.3435],
         [-0.0531,  1.0216, -0.2858,  ..., -0.1946,  0.3140,  0.2594],
         [ 0.4319, -0.5107,  0.7150,  ..., -0.3964,  0.4322, -0.0792],
         ...,
         [ 1.7345,  0.8501, -0.4626,  ..., -0.2404,  0.8107,  1.1856],
         [-1.7627, -1.0018,  1.2368,  ...,  0.7324, -1.3661,  0.5996],
         [-0.9521, -0.4527, -0.6780,  ..., -1.6143,  0.4245, -1.4192]],

        [[ 1.0265, -0.5188, -0.1022,  ...,  0.4857, -0.2457, -0.3435],
         [-0.0531,  1.0216, -0.2858,  ..., -0

In [149]:
edge_mask.shape, edge_mask.sum() / edge_mask.numel()

(torch.Size([44, 44]), tensor(0.3683))